## Modelo XGBoost

### Carga de librerías

In [11]:
import polars as pl
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import log_loss
import xgboost as xgb
from pathlib import Path

### Carga de datos

In [12]:
ROOT = Path("..")
data_path = ROOT / "data" / "processed" / "temporada1_limpio.parquet"
df = pl.read_parquet(data_path).to_pandas()

Vamos a chequear las columnas que tiene el dataset de la temporada 2. Las columnas a incluir en el modelo XGBoost van a ser todas las que contenga el dataset de la temporada 2 más las que se calculan a partir de ellas (*feature engineering*)

In [13]:
ruta_t2 = ROOT / "data" / "raw" / "temporada2.parquet"

df_t2 = pl.read_parquet(ruta_t2)

cols_t1 = set(df.columns)
cols_t2 = set(df_t2.columns)

# Buscamos diferencias (ignorando la variable 'description' o 'swing' que lógicamente no va a estar en la T2)
faltan_en_t2 = cols_t1 - cols_t2
sobran_en_t2 = cols_t2 - cols_t1

print(f"Columnas que están en T1 pero FALTAN en T2: {faltan_en_t2}")
print(f"Columnas que están en T2 pero NO ESTABAN en T1: {sobran_en_t2}")

Columnas que están en T1 pero FALTAN en T2: {'platoon_advantage', 'strike_zone_size', 'description', 'distance_to_corner', 'is_shadow_zone', 'sz_mid', 'relative_x', 'relative_height', 'complex_speed', 'is_strike_zone', 'pitch_location', 'swing', 'movement_complexity', 'count'}
Columnas que están en T2 pero NO ESTABAN en T1: set()


La única diferencia entre los archivos .parquet de las temporadas 1 y 2 son las variables *swing* y *description* (variable respuesta) y todas las que calculamos en el *feature engineering*.

### Definición de variables predictoras y respuesta

In [14]:
columnas_excluir = ['pitch_id', 'batter', 'pitcher', 'description', 'swing']
explicativas = df.drop(columns=[col for col in columnas_excluir if col in df.columns])
respuesta = df['swing']

# A XGBoost hay que definirle cuáles son las variables categóricas
columnas_texto = explicativas.select_dtypes(include=['object', 'string']).columns
for col in columnas_texto:
    explicativas[col] = explicativas[col].astype('category')

print(f"El dataset tiene {explicativas.shape[0]} filas y {explicativas.shape[1]} variables predictoras.")

El dataset tiene 704721 filas y 24 variables predictoras.


### Configuración del modelo XGBoost

In [16]:
xgb_base = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    enable_categorical=True, # Avisa que hay columnas categóricas
    tree_method='hist',      # Algoritmo de histogramas, rápido para datasets grandes
    random_state=42
)

Vamos a definir una serie de hiperparámetros distintos y vamos a probar solo 20 al azar. Queda para otro momento el analizar otras técnicas de selección de hiperparámetros.

In [ ]:
param_dist = {
    'max_depth': [3, 5, 7],               # Profundidad del árbol
    'learning_rate': [0.01, 0.05, 0.1],   # Qué tan rápido aprende
    'n_estimators': [100, 300, 500],      # Cantidad de árboles
    'subsample': [0.8, 1.0],              # Porcentaje de filas a usar por árbol
    'colsample_bytree': [0.8, 1.0]        # Porcentaje de columnas a usar por árbol
}

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42) # Usamos 3 folds para que sea más rápido
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=20, # límite de combinaciones de hiperparámetros a probar
    scoring='neg_log_loss', # Scikit-learn busca maximizar, por eso usa LogLoss negativo
    cv=skf,
    verbose=1, # Te va mostrando el progreso en la consola
    random_state=42
)

random_search.fit(explicativas, respuesta)
print(f"Mejores hiperparámetros: {random_search.best_params_}")
print(f"Mejor LogLoss estimado: {-random_search.best_score_: .4f}")

mejor_modelo = random_search.best_estimator_

### Predicción para Kaggle

El primer paso es crear las variables del *feature engineering* en el dataset de la temporada 2.

Respecto al dataset de la temporada 1, acá vamos a omitir eliminar las filas con celdas vacías porque debemos respetar la cantidad de filas totales que tiene el dataset de la temporada 2.

In [ ]:
# Constantes de conversión
FT_TO_CM = 30.48
HALF_PLATE_CM = 21.59   
SHADOW_MARGIN_CM = 5.08  

data_t2 = (
    df_t2
    .with_columns( # Conversión de unidades (pies a cm)
        plate_x = pl.col("plate_x") * FT_TO_CM,
        plate_z = pl.col("plate_z") * FT_TO_CM,
        pfx_x   = pl.col("pfx_x")   * FT_TO_CM,
        pfx_z   = pl.col("pfx_z")   * FT_TO_CM,
        sz_top  = pl.col("sz_top")  * FT_TO_CM,
        sz_bot  = pl.col("sz_bot")  * FT_TO_CM
    )
    .with_columns( # Métricas base de zona y movimiento 
        sz_mid = (pl.col("sz_top") + pl.col("sz_bot")) / 2, 
        strike_zone_size = pl.col("sz_top") - pl.col("sz_bot"), 
        movement_complexity = (pl.col("pfx_x")**2 + pl.col("pfx_z")**2).sqrt(), 
        platoon_advantage = (pl.col("p_throws") != pl.col("stand")).cast(pl.Int32) 
    )
    .with_columns( # Localización normalizada
        relative_height = ( 
            (pl.col("plate_z") - pl.col("sz_bot")) / pl.col("strike_zone_size")
        ),
        relative_x = pl.col("plate_x") / HALF_PLATE_CM 
    )
    .with_columns( # Indicadoras de zona de strike
        is_strike_zone = (
            pl.col("plate_x").is_between(-HALF_PLATE_CM, HALF_PLATE_CM) &
            pl.col("relative_height").is_between(0, 1)
        ).cast(pl.Int32)
    )
    .with_columns( # Indicadoras de zona de sombra
        is_shadow_zone = (
            pl.col("plate_x").is_between(
                -HALF_PLATE_CM - SHADOW_MARGIN_CM,
                 HALF_PLATE_CM + SHADOW_MARGIN_CM
            ) &
            pl.col("relative_height").is_between(-0.1, 1.1) &
            (pl.col("is_strike_zone") == 0)
        ).cast(pl.Int32)
    )
    .with_columns( # Distancia al centro de zona
        pitch_location = (
            pl.col("plate_x")**2 +
            (pl.col("plate_z") - pl.col("sz_mid"))**2
        ).sqrt(),
    )
    .with_columns(# Distancia al rincón más cercano 
    distance_to_corner = pl.when(pl.col("is_strike_zone") == 1).then(
        (
            (1 - pl.col("relative_x").abs())**2 +
            pl.min_horizontal(
                pl.col("relative_height"),        
                1 - pl.col("relative_height"),    
            )**2
        ).sqrt()
    ).otherwise(pl.lit(0.0))
    )
    .with_columns( # Interacciones
        complex_speed = pl.col("release_speed") * pl.col("movement_complexity"), 
        count = pl.col("balls").cast(pl.Utf8) + pl.lit("-") + pl.col("strikes").cast(pl.Utf8) 
    )
)

# Guardamos el archivo procesado listo para el predict
ruta_salida = ROOT / 'data' / 'processed' / 'temporada2_limpio.parquet'
data_t2.write_parquet(file=ruta_salida)

In [ ]:
# ¡OJO ACÁ! Tenés que leer la Temporada 2 que ya pasó por el mismo Feature Engineering que la 1
ruta_t2_procesada = ROOT / "data" / "processed" / "temporada2_limpio.parquet" 

df_t2 = pl.read_parquet(ruta_t2_procesada).to_pandas()

# Guardamos los IDs para el archivo final (Kaggle siempre pide el ID)
pitch_ids_t2 = df_t2['pitch_id']

# Filtramos X_test para que tenga EXACTAMENTE las mismas columnas que usamos para entrenar
X_test = df_t2[X.columns]

# Convertimos las columnas de texto a categóricas, igual que en el entrenamiento
columnas_texto = X_test.select_dtypes(include=['object', 'string']).columns
for col in columnas_texto:
    X_test[col] = X_test[col].astype('category')

# Predecimos las probabilidades
print("Calculando probabilidades de Swing...")
probabilidades_swing = mejor_modelo.predict_proba(X_test)[:, 1]

# Armamos el DataFrame final
submission = pd.DataFrame({
    'pitch_id': pitch_ids_t2,
    'swing': probabilidades_swing
})

In [ ]:
# Guardamos el CSV
ruta_submission = ROOT / "reports" / "mi_primera_submission.csv"
submission.to_csv(ruta_submission, index=False)
print(f"¡Éxito! Archivo guardado en: {ruta_submission}")